# Study 04 ENNI Context Ablation

This notebook runs the `utterance_only` vs `prev_same_speaker` ENNI context-window experiment from this repo.

Workflow:
1. Clone your repo branch.
2. Install inference dependencies.
3. Set your Hugging Face token.
4. Build the ENNI evaluation JSONL.
5. Run the model in two modes.
6. Compare the outputs.


In [ ]:
REPO_URL = "https://github.com/<YOUR-USER>/<YOUR-REPO>.git"
BRANCH = "<YOUR-BRANCH>"
REPO_DIR = "/content/talkbank-morphosyntax-error-annotator"

# Set LIMIT = 50 for a smoke test, then switch to 0 for the full run.
LIMIT = 50
BATCH_SIZE = 4
OUTPUT_DIR = "study_04_context_windows/results/enni_context_ablation"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!git status --short


In [ ]:
!nvidia-smi
!python3 --version


In [ ]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece scipy orjson


In [ ]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/prepare_enni_context_ablation_input.py


Run the frozen model on the same ENNI JSONL in `utterance_only` mode.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/run_ood_context_inference.py \
  --input-jsonl study_04_context_windows/data/context_eval/enni_reviewed_prev_same_speaker_eval.jsonl \
  --context-mode utterance_only \
  --out-dir "$OUTPUT_DIR" \
  --batch-size "$BATCH_SIZE" \
  --limit "$LIMIT"


Run the same JSONL again in `prev_same_speaker` mode.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/run_ood_context_inference.py \
  --input-jsonl study_04_context_windows/data/context_eval/enni_reviewed_prev_same_speaker_eval.jsonl \
  --context-mode prev_same_speaker \
  --out-dir "$OUTPUT_DIR" \
  --batch-size "$BATCH_SIZE" \
  --limit "$LIMIT"


Compare the two prediction files. The main summary is written to `analysis/summary.json`.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/analyze_enni_context_ablation.py \
  --utterance-only "$OUTPUT_DIR/predictions_utterance_only.jsonl" \
  --prev-same-speaker "$OUTPUT_DIR/predictions_prev_same_speaker.jsonl" \
  --out-dir "$OUTPUT_DIR/analysis"


In [ ]:
import json
from pathlib import Path

summary_path = Path(REPO_DIR) / OUTPUT_DIR / "analysis" / "summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


For the full experiment, set `LIMIT = 0` in the config cell and rerun the two inference cells plus the analysis cell.
